
# Blending star-forming galaxy and AGN accretion disc continua

Active galactic nuclei dominate UV to infrared SEDs. Sweeps AGN
luminosity fraction from pure starburst to pure AGN, showing the
transition in SED morphology as the accretion disc continuum
increasingly dominates stellar and dust emission.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# Build composite model: stellar + AGN with sweepable AGN luminosity
ssp = tengri.load_ssp()
model = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "dpl",
        "all_params": tengri.FIXED,
        "alpha": 2.0,
        "beta": 2.5,
        "tau_gyr": 1.0,
        "log_total_mass": 10.0,
    },
    dust={
        "type": "two_component",
        "all_params": tengri.FIXED,
        "tau_bc": 0.3,
        "tau_diff": 0.2,
        "emission": {"type": "dale2014", "all_params": tengri.FIXED},
    },
    agn={
        "type": "composable",
        "disc": {"type": "qsogen", "all_params": tengri.FIXED},
    },
    redshift=tengri.Fixed(0.05),
)

# Base parameters
baseline = dict(model.spec.sample(jax.random.PRNGKey(42)))

# AGN luminosity values to sweep (negative means no AGN contribution, positive means AGN-dominated)
# Sweep log_lbol from -2 (negligible AGN) to 11 (strong AGN)
agn_log_lbols = np.array([0.0, 9.0, 10.0, 10.5, 11.0, 11.5])
# Normalize to fraction for colorbar (0 = no AGN, 1 = strong AGN)
agn_fracs = (agn_log_lbols - agn_log_lbols.min()) / (agn_log_lbols.max() - agn_log_lbols.min())

cmap = plt.get_cmap("viridis")
norm = plt.Normalize(vmin=0, vmax=1)

fig, ax = plt.subplots(figsize=(10, 5.2))

for agn_log_lbol, agn_frac in zip(agn_log_lbols, agn_fracs):
    # Update the AGN log luminosity parameter
    params = dict(baseline)
    params["agn_log_lbol"] = agn_log_lbol

    # Predict composite SED
    out = model.predict(params)
    wave = np.asarray(model.wavelengths)
    sed = np.asarray(out.rest_sed())
    wave_um = wave / 1e4

    # Compute nu * L_nu for plotting
    nu = 2.998e18 / wave
    nu_l_nu = nu * sed

    mask = sed > 0
    ax.loglog(
        wave_um[mask],
        nu_l_nu[mask],
        color=cmap(norm(agn_frac)),
        lw=2.0,
    )

ax.set_xlim(0.08, 1e2)
ax.set_ylim(1e40, 1e45)
ax.set_xlabel(r"Rest-frame wavelength $\lambda$ [$\mu$m]")
ax.set_ylabel(r"$\nu L_\nu$ [erg s$^{-1}$]")

cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, pad=0.01)
cbar.set_label(r"AGN luminosity fraction $f_{\rm AGN}$")

fig.tight_layout()
plt.savefig("plot_panchromatic_agn_fraction.png", dpi=150, bbox_inches="tight")